# ShadowKV++ Semantic Fidelity — Full 5-Model, 10-Dataset Evaluation

**Measures whether splicing cached KV from one prompt prefix into another's computation changes the generated output.**

Uses `DynamicCache.crop()` (transformers 5.x) for zero-overhead KV cache splicing.
The first 75% of prompt tokens are shared (identical), the last 25% are shuffled
to simulate a semantic-approximate match.

**5 models**: GPT-2, TinyLlama, Qwen2.5 1.5B, Gemma 2B, Phi-3 Mini
**10 datasets**: samsum, xsum, cnn_dailymail, ag_news, banking77, alpaca_eval, dolly, daily_dialog, oasst1, ultrachat

Expected: **KV fidelity = 1.0** for LLaMA/Gemma/GPT-2/Phi-3, **~0.99** for Qwen2.

In [ ]:
# Install deps
!pip install -q torch transformers datasets accelerate huggingface-hub rouge-score numpy pyarrow pandas

from huggingface_hub import login
login(token='YOUR_HF_TOKEN_HERE')

!git clone https://github.com/Kushalk0677/shadowkv.git
%cd shadowkv/experiments

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
%%writefile run_fidelity_equiv.py
import argparse, json, os, sys, time, random, warnings
warnings.filterwarnings('ignore', category=UserWarning, module='transformers')
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELS = {
    'gpt2': 'gpt2',
    'tinyllama': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'qwen25_15b': 'Qwen/Qwen2.5-1.5B-Instruct',
    'gemma2b': 'google/gemma-2b-it',
    'phi3mini': 'microsoft/Phi-3-mini-4k-instruct',
}

DATASETS = {
    'samsum': ('knkarthick/samsum', 'train', 'dialogue'),
    'xsum': ('xsum', 'train', 'document'),
    'cnn_dailymail': ('abisee/cnn_dailymail', '3.0.0', 'train', 'article'),
    'ag_news': ('fancyzhx/ag_news', 'train', 'text'),
    'banking77': ('mteb/banking77', 'train', 'text'),
    'alpaca_eval': ('Thanmay/alpaca_eval', 'eval', 'instruction'),
    'dolly': ('databricks/databricks-dolly-15k', 'train', 'instruction'),
    'daily_dialog': ('DeepPavlov/daily_dialog', 'train', 'dialog'),
    'oasst1': ('OpenAssistant/oasst1', 'train', 'text'),
    'ultrachat': ('HuggingFaceH4/ultrachat_200k', 'train_sft', 'messages'),
}

PARQUET_DS = {
    'xsum': ('xsum', 'data/train-00000-of-00001.parquet', 'document'),
}

def load_model(m, d):
    t = AutoTokenizer.from_pretrained(m)
    if t.pad_token is None: t.pad_token = t.eos_token
    return t, AutoModelForCausalLM.from_pretrained(m, torch_dtype=torch.float16, low_cpu_mem_usage=True).to(d).eval()

def gen_std(model, ids, mn):
    with torch.no_grad():
        o = model.generate(ids, max_new_tokens=mn, do_sample=False, num_beams=1, pad_token_id=model.config.pad_token_id or 0)
    return o[0][ids.shape[1]:]

def gen_reuse(model, oi, mi, sh, mn, eos):
    sh = min(sh, len(oi[0]), len(mi[0]))
    with torch.no_grad():
        ca = model(oi, use_cache=True).past_key_values
    ca.crop(sh)
    sf = mi[:, sh:]
    if sf.shape[1] > 0:
        with torch.no_grad():
            cc = model(sf, past_key_values=ca, use_cache=True).past_key_values
        tp, st = sh + sf.shape[1], sf[:, -1:]
    else:
        cc, tp, st = ca, sh, oi[:, sh-1:sh]
    if tp < 1: return gen_std(model, mi, mn)
    cc.crop(tp-1)
    inp, kv, g = st, cc, []
    for _ in range(mn):
        with torch.no_grad():
            o = model(input_ids=inp, past_key_values=kv, use_cache=True)
        n = o.logits[0,-1].argmax(-1, keepdim=True).unsqueeze(0)
        inp, kv = n, o.past_key_values
        g.append(n.item())
        if n.item() == eos: break
    return torch.tensor(g)

def shuffle_ids(ids, r=0.75):
    sp = int(len(ids)*r)
    if len(ids)-sp < 4: sp = len(ids)-4
    if sp < 4: return ids, len(ids)
    return ids[:sp] + random.sample(ids[sp:], len(ids[sp:])), sp

def extract(row, k):
    if k in ('alpaca_eval','dolly'): return str(row.get('instruction',''))
    if k == 'samsum': return str(row.get('dialogue',''))
    if k == 'xsum': return str(row.get('document',''))
    if k == 'cnn_dailymail': return str(row.get('article',''))
    if k in ('ag_news','banking77'): return str(row.get('text',''))
    if k == 'daily_dialog':
        d = row.get('dialog',[])
        return ' '.join(d) if isinstance(d,list) else str(d)
    if k == 'oasst1': return str(row.get('text',''))
    if k == 'ultrachat':
        ms = row.get('messages',[])
        return ' '.join(m.get('content','') for m in ms if isinstance(m,dict))
    return str(row.get(list(row.keys())[0],''))

def load_samp(dk, n):
    if dk in PARQUET_DS:
        import pyarrow.parquet as pq
        from huggingface_hub import hf_hub_download
        r, pf, f = PARQUET_DS[dk]
        t = pq.read_table(hf_hub_download(repo_id=r, filename=pf, repo_type='dataset'))
        return [str(t.column(f)[i].as_py() or '')[:512] for i in range(min(n,len(t))) if len(str(t.column(f)[i].as_py() or '')) > 20]
    from datasets import load_dataset
    info = DATASETS[dk]
    if len(info) == 4:
        ds = load_dataset(info[0], info[1], split=info[2], trust_remote_code=True)
    else:
        ds = load_dataset(info[0], split=info[1], trust_remote_code=True)
    return [extract(ds[i], dk)[:512] for i in range(min(n,len(ds))) if len(extract(ds[i], dk)) > 20]

def run(args):
    import json
    od = Path(args.output_dir); od.mkdir(parents=True, exist_ok=True)
    ar = []; random.seed(42)
    for mk in args.models:
        tok, model = load_model(MODELS[mk], args.device)
        eos = tok.eos_token_id or 0
        for dk in args.datasets:
            print(f'\n=== {mk} on {dk} ===')
            sa = load_samp(dk, args.n_samples)
            print(f'  {len(sa)} samples')
            if not sa: continue
            for payload in sa:
                i1 = tok.encode(payload, truncation=True, max_length=384)
                if len(i1) < 16: continue
                for ra in args.ratios:
                    m1, sh = shuffle_ids(i1, r=ra)
                    io = torch.tensor([i1], device=args.device)
                    im = torch.tensor([m1], device=args.device)
                    ex = tok.decode(gen_std(model, io, args.max_gen_tokens), skip_special_tokens=True)
                    rf = tok.decode(gen_std(model, im, args.max_gen_tokens), skip_special_tokens=True)
                    ru = tok.decode(gen_reuse(model, io, im, sh, args.max_gen_tokens, eos), skip_special_tokens=True)
                    ar.append({'model':mk,'dataset':dk,'shared_ratio':ra,'shared_tokens':sh,
                        'total_tokens_orig':len(i1),'exact_text':ex,'ref_text':rf,'reuse_text':ru})
        with open(od/f'{mk}_results.json','w') as f:
            json.dump([x for x in ar if x['model']==mk], f, indent=2)
        print(f'Saved -> {od}/{mk}_results.json')
        del model, tok
        if args.device.startswith('cuda'): torch.cuda.empty_cache()
    with open(od/'all_results.json','w') as f: json.dump(ar, f, indent=2)
    print(f'Total: {len(ar)}')

if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('--device', default='cuda:0')
    p.add_argument('--models', nargs='+', default=['tinyllama'], choices=MODELS.keys())
    p.add_argument('--datasets', nargs='+', default=['samsum','alpaca_eval','banking77'], choices=DATASETS.keys())
    p.add_argument('--n_samples', type=int, default=32)
    p.add_argument('--max_gen_tokens', type=int, default=64)
    p.add_argument('--ratios', nargs='+', type=float, default=[0.75])
    p.add_argument('--output_dir', default='fidelity_results')
    args = p.parse_args()
    run(args)

In [ ]:
# Run — 5 models x 10 datasets x 32 samples (~1 hour on T4)
!python run_fidelity_equiv.py \
  --device cuda:0 \
  --models gpt2 tinyllama qwen25_15b gemma2b phi3mini \
  --datasets samsum xsum cnn_dailymail ag_news banking77 alpaca_eval dolly daily_dialog oasst1 ultrachat \
  --n_samples 32 \
  --max_gen_tokens 64 \
  --output_dir fidelity_results

In [ ]:
# Evaluate
from rouge_score import rouge_scorer
import json

r = json.load(open('fidelity_results/all_results.json'))
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

matches = sum(1 for x in r if x['ref_text'] == x['reuse_text'])
print(f'KV Fidelity: {matches}/{len(r)} exact match ({matches/len(r)*100:.1f}%)')

print(f'\nBy model:')
by_model = {}
for x in r:
    by_model.setdefault(x['model'], []).append(x)
for mk, items in sorted(by_model.items()):
    m = sum(1 for x in items if x['ref_text'] == x['reuse_text'])
    n = len(items)
    ps = sum(scorer.score(x['exact_text'], x['ref_text'])['rougeL'].fmeasure
             for x in items if x['exact_text'] and x['ref_text']) / max(n, 1)
    print(f'  {mk:>12}: {m}/{n} KV hits ({m/n*100:5.1f}%), prompt sens ROUGE={ps:.4f}')

print(f'\nBy dataset:')
by_ds = {}
for x in r:
    by_ds.setdefault(x['dataset'], []).append(x)
for dk, items in sorted(by_ds.items()):
    m = sum(1 for x in items if x['ref_text'] == x['reuse_text'])
    print(f'  {dk:>15}: {m}/{len(items)} KV hits ({m/len(items)*100:5.1f}%)')

In [ ]:
# Download
from google.colab import files
!zip -r /content/fidelity_results.zip fidelity_results
files.download('/content/fidelity_results.zip')